## Create your first unity catalog and schema 

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS my_first_catalog
MANAGED LOCATION 'abfss://delta@templearningdelta.dfs.core.windows.net/'

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS my_first_catalog.my_first_schema;

In [0]:
%sql
USE CATALOG my_first_catalog;
SHOW SCHEMAS;

### Import PySpark libraries and DataFrame functions.

In [0]:
from delta.tables import *
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, array, ArrayType, DateType, TimestampType, FloatType, DoubleType, DecimalType
from pyspark.sql.functions import *

## Create a Pyspark dataframe

In [0]:
employee_data = [
    {"emp_id": 1, "emp_name": "John Sean", "dept_id": 101, "salary": 60000.00, "hire_date": "2025-01-15"},
    {"emp_id": 2, "emp_name": "Alice Cooper", "dept_id": 102, "salary": 75000.00, "hire_date": "2025-03-20"},
    {"emp_id": 3, "emp_name": "Bob K.", "dept_id": 101, "salary": 50000.00, "hire_date": "2025-07-10"},
    {"emp_id": 4, "emp_name": "David Harbour", "dept_id": 103, "salary": 82000.00, "hire_date": "2026-02-05"},
    {"emp_id": 5, "emp_name": "Emma Norton", "dept_id": 102, "salary": 67000.00, "hire_date": "2026-01-25"},
    {"emp_id": 6, "emp_name": "Frank Jeff", "dept_id": 103, "salary": 70000.00, "hire_date": "2025-04-15"},
    {"emp_id": 7, "emp_name": "Grace Morgan", "dept_id": 104, "salary": 55000.00, "hire_date": "2026-01-09"},
    {"emp_id": 8, "emp_name": "Henry J. Knott", "dept_id": 104, "salary": 60000.00, "hire_date": "2025-06-03"},
    {"emp_id": 9, "emp_name": "Ivy Lee", "dept_id": 101, "salary": 72000.00, "hire_date": "2025-09-18"},
    {"emp_id": 10, "emp_name": "Jack White", "dept_id": 103, "salary": 65000.00, "hire_date": "2026-02-08"},
]

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("dept_id", IntegerType(), True),
    StructField("salary", DoubleType(), True),
    StructField("hire_date", StringType(), True)
])

df_employees = spark.createDataFrame(employee_data, schema=schema)

df_employees = df_employees.withColumn("hire_date", to_date("hire_date"))

df_employees.printSchema()
df_employees.show()


## Create a Delta Table and apply partitioning to optimize query performance

In [0]:
%sql
DROP TABLE IF EXISTS my_first_catalog.my_first_schema.employees;

In [0]:
%sql
CREATE TABLE my_first_catalog.my_first_schema.employees (
    emp_id INT PRIMARY KEY,
    emp_name STRING,
    dept_id INT,
    salary DOUBLE,
    hire_date DATE
) USING DELTA
PARTITIONED BY (dept_id);

In [0]:
df_employees.write.format("delta").mode("append").saveAsTable("my_first_catalog.my_first_schema.employees")

### Check Delta table data and details

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees LIMIT 5;

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.employees;

### Check Delta table history

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.employees;


### Recreate table with with a generated column 

In [0]:
%sql
CREATE TABLE my_first_catalog.my_first_schema.employees_gen (
    emp_id INT,
    emp_name STRING,
    dept_id INT,
    salary DOUBLE,
    hire_date DATE,
    hire_year INT GENERATED ALWAYS AS (YEAR(hire_date))
) USING DELTA
PARTITIONED BY (dept_id);

In [0]:
df_employees.write.format("delta").mode("append").saveAsTable("my_first_catalog.my_first_schema.employees_gen")

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees_gen LIMIT 5;

### Perform DML operations in Delta table - INSERT

In [0]:
%sql
INSERT INTO my_first_catalog.my_first_schema.employees
VALUES (11, 'Chris Doe', 104, 55000,  '2024-01-10');


In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=11; 

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.employees;

### Perform DML operations in Delta table - UPDATE


In [0]:
spark.conf.set(
    "spark.databricks.delta.commitInfo.userMetadata",
    "This row was updated because of a salary change of the employee."
)

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=5; 

In [0]:
%sql
UPDATE my_first_catalog.my_first_schema.employees SET salary=90000 WHERE emp_id=5;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.employees;

### Perform DML operations in Delta table - DELETE 

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=5;

In [0]:
%sql
DELETE FROM my_first_catalog.my_first_schema.employees WHERE emp_id=5;
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=5;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.employees;

### Perform time-travel in Delta table 

In [0]:
%%sql
SELECT * FROM my_first_catalog.my_first_schema.employees VERSION AS OF 3 WHERE emp_id=5;

In [0]:
%%sql
SELECT * FROM my_first_catalog.my_first_schema.employees TIMESTAMP AS OF '2026-02-20 21:31:13' WHERE emp_id=5;

In [0]:
%sql
RESTORE TABLE my_first_catalog.my_first_schema.employees TO VERSION AS OF 3;
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=5;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.employees;

## Perform CDC merge in Delta table

In [0]:
employee_cdc_data = [
    {"emp_id": 2, "emp_name": "Alice Cooper", "dept_id": 102, "salary": 80000.00, "hire_date": "2025-03-20"},
    {"emp_id": 11, "emp_name": "Jack Mathew", "dept_id": 104, "salary": 50000.00, "hire_date": "2026-02-10"},
    {"emp_id": 12, "emp_name": "Kristen Johnson", "dept_id": 103, "salary": 45000.00, "hire_date": "2025-02-10"}
]

df_employees_cdc = spark.createDataFrame(employee_cdc_data, schema)
df_employees_cdc = df_employees_cdc.withColumn("hire_date", to_date("hire_date"))
display(df_employees_cdc)
df_employees_cdc.createOrReplaceTempView("cdc_employees")


In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=2;

In [0]:
%sql
SELECT count(*) FROM my_first_catalog.my_first_schema.employees;

In [0]:
%sql
MERGE INTO my_first_catalog.my_first_schema.employees
USING cdc_employees
ON cdc_employees.emp_id = my_first_catalog.my_first_schema.employees.emp_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=2
UNION
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=11
UNION
SELECT * FROM my_first_catalog.my_first_schema.employees WHERE emp_id=12;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.employees;

## Perform a CDC merge in Delta Lake with schema evolution

In [0]:
cdc_data1 = [
    {"emp_id": 13, "emp_name": "Jane Marry", "dept_id": 102, "salary": 85000.00, "hire_date": "2025-02-22", "birth_date": "1968-02-12"},
    {"emp_id": 14, "emp_name": "Andrew Anderson", "dept_id": 103, "salary": 55000.00, "hire_date": "2026-02-11", "birth_date": "1971-06-02"},
    {"emp_id": 15, "emp_name": "Carl Degz", "dept_id": 102, "salary": 75000.00, "hire_date": "2026-02-11", "birth_date": "1976-12-11"}
]

df_employees_cdc1 = spark.createDataFrame(cdc_data1)
df_employees_cdc1 = df_employees_cdc1.withColumn("hire_date", to_date("hire_date"))
df_employees_cdc1.show()
df_employees_cdc1.printSchema()
df_employees_cdc1.createOrReplaceTempView("cdc_employees_1")

In [0]:
spark.sql ('select * from cdc_employees_1').show()

In [0]:
%sql
MERGE WITH SCHEMA EVOLUTION INTO my_first_catalog.my_first_schema.employees
USING cdc_employees_1
ON cdc_employees_1.emp_id = my_first_catalog.my_first_schema.employees.emp_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees;

## Explicitly modify schema of Delta table

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.employees ADD COLUMNS (phone STRING AFTER emp_name);

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees LIMIT 2;

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.employees SET TBLPROPERTIES ('delta.columnMapping.mode' ='name');
ALTER TABLE my_first_catalog.my_first_schema.employees RENAME COLUMN phone TO cell_phone;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees LIMIT 2;

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.employees DROP COLUMN cell_phone;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees LIMIT 2;

## Use the replaceWhere feature in Delta Lake

In [0]:
replace_data = (
    spark.table("my_first_catalog.my_first_schema.employees")                             
         .filter("birth_date IS NULL")   
         .withColumn("birth_date", lit("unknown"))    
)

In [0]:
spark.conf.set("spark.databricks.delta.replaceWhere.constraintCheck.enabled", False)
(replace_data.write
  .mode("overwrite")
  .option("replaceWhere", "birth_date IS NULL")
  .saveAsTable("my_first_catalog.my_first_schema.employees")
)

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.employees;

## Set up a NOT NULL constraint in Delta Table

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.employees
ALTER COLUMN emp_id SET NOT NULL;

In [0]:
%sql
INSERT INTO my_first_catalog.my_first_schema.employees
VALUES (NULL, 'Chris', 104, 55000.00,  '2024-01-10',  '2076-05-10');
    

## Set up a CHECK constraint in Delta Table

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.employees ADD CONSTRAINT valid_date CHECK (hire_date >= '2025-01-01' and hire_date < '2027-01-01');

In [0]:
%sql
INSERT INTO my_first_catalog.my_first_schema.employees (emp_id,emp_name,dept_id,salary,hire_date, birth_date) VALUES (15,'Chris',104,55000.00,'2027-01-10', '1997-01-01');

## Set up a foreign key constraint in Delta table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS my_first_catalog.my_first_schema.department (dept_id INT PRIMARY KEY, dept_name VARCHAR(255), dept_head VARCHAR(255));


In [0]:
%sql
INSERT INTO my_first_catalog.my_first_schema.department VALUES (101,'Sales','John Doe'),(102,'Marketing','Jane Smith'),(103,'Engineering','Alice Johnson'),(104,'HR','Bob Brown');
    


In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.employees ADD CONSTRAINT dept_id FOREIGN KEY (dept_id) REFERENCES my_first_catalog.my_first_schema.department (dept_id);

In [0]:
%sql
INSERT INTO my_first_catalog.my_first_schema.employees(emp_id, emp_name, dept_id, salary, hire_date, birth_date) VALUES (18,'May Lauren',106,60000,'2025-01-01', '1997-01-01');

%md
##  Optimize Delta table

In [0]:
%sql
OPTIMIZE  my_first_catalog.my_first_schema.employees;

##  Optimize using Z-ordering in Delta table

In [0]:
%sql
OPTIMIZE  my_first_catalog.my_first_schema.employees ZORDER BY (emp_name, hire_date);

## Optimize using liquid clustering in Delta table

In [0]:
df_employees.write \
  .format("delta") \
  .clusterBy("dept_id", "hire_date") \
  .saveAsTable("my_first_catalog.my_first_schema.employees_clustered")

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.employees_clustered;

## Regularly clean old Delta files

In [0]:
%sql
VACUUM  my_first_catalog.my_first_schema.employees;

### Isolation levels in Delta table


In [0]:
%sql
SELECT version, operation, isolationLevel
from (DESCRIBE HISTORY my_first_catalog.my_first_schema.employees);

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.employees SET TBLPROPERTIES ('delta.isolationLevel' = 'Serializable')

In [0]:
%sql
UPDATE my_first_catalog.my_first_schema.employees SET salary=95000 WHERE emp_id=5;

In [0]:
%sql
SELECT version, operation, isolationLevel
from (DESCRIBE HISTORY my_first_catalog.my_first_schema.employees);